# Phase 1 — Data preparation

Load and clean the SolarEdge data, build model features, and define the target.


## Setup


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pvlib") is None:
    print(f"Installing pvlib into {sys.executable} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pvlib"])

from pathlib import Path

import numpy as np
import pandas as pd
import pvlib

SITE = dict(latitude=22.3364, longitude=114.2654, altitude=60)
TIMEZONE = "Asia/Hong_Kong"

EXPERIMENT_ROOT = Path("model d")
if not EXPERIMENT_ROOT.exists():
    EXPERIMENT_ROOT = Path(".")

LOCAL_INPUT_PATH = (
    EXPERIMENT_ROOT / "dataset" / "site_level_dataset_modified_combined.csv"
)
COLAB_INPUT_PATH = Path("/content/site_level_dataset_modified_combined.csv")
INPUT_PATH = (
    LOCAL_INPUT_PATH if LOCAL_INPUT_PATH.exists() else COLAB_INPUT_PATH
)
OUTPUT_PATH = EXPERIMENT_ROOT / "dataset" / "solaredge_clean.parquet"

GHI_COL = "Irradiance (W/m2)"
TEMP_COL = "Temp (Degree Celsius)"
WIND_COL = "Wind Speed (m/s)"
RH_COL = "RH (%)"
POWER_COL = "power(W)"
GROUP_COL = "station"
TARGET_COL = "y_norm"

WEATHER_FEATURES = [GHI_COL, TEMP_COL, WIND_COL, RH_COL]
PHYSICS_FEATURES = [
    "zenith", "azimuth", "clearsky_ghi",
    "clearsky_index", "cell_temp_ghi",
]
MODEL_COLUMNS = WEATHER_FEATURES + PHYSICS_FEATURES + [TARGET_COL]

EXPECTED_INTERVAL = pd.Timedelta(minutes=15)
STUCK_MIN_SAMPLES = 16  # 4 hours at 15-minute resolution
FAULT_ZERO_GHI = 200.0


## Load data


In [ ]:
df = pd.read_csv(INPUT_PATH, parse_dates=["Time"]).rename(
    columns={"Time": "timestamp"}
)
if df["timestamp"].dt.tz is None:
    df["timestamp"] = df["timestamp"].dt.tz_localize(TIMEZONE)

df = df.sort_values([GROUP_COL, "timestamp"]).reset_index(drop=True)
assert not df.duplicated([GROUP_COL, "timestamp"]).any(),             "Duplicate (station, timestamp) rows found"

df["gap_from_previous_min"] = (
    df.groupby(GROUP_COL, sort=False)["timestamp"]
      .diff().dt.total_seconds().div(60)
)
df["flag_after_gap"] = df["gap_from_previous_min"].gt(
    EXPECTED_INTERVAL.total_seconds() / 60
)

n_systems = df[GROUP_COL].nunique()
assert n_systems == 37, f"Expected 37 systems in the combined table, found {n_systems}"
assert df["total_capacity_W"].notna().all(), "Missing system capacity in combined table"
assert df["total_capacity_W"].gt(0).all(), "System capacity must be positive"
assert df.groupby(GROUP_COL)["total_capacity_W"].nunique().eq(1).all(),             "A station has inconsistent capacity values in the combined table"

print(f"Loaded {len(df):,} rows from {n_systems} systems.")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Raw data-gap events: {df['flag_after_gap'].sum():,}")


## Solar features


In [ ]:
location = pvlib.location.Location(tz=TIMEZONE, **SITE)
unique_times = pd.DatetimeIndex(df["timestamp"].unique()).sort_values()
solar_position = location.get_solarposition(unique_times)

solar_features = pd.DataFrame({
    "timestamp": unique_times,
    "zenith": solar_position["apparent_zenith"].to_numpy(),
    "azimuth": solar_position["azimuth"].to_numpy(),
})
df = df.merge(solar_features, on="timestamp", how="left", validate="many_to_one")

rows_before_daylight = len(df)
df = df.loc[df["zenith"].lt(90)].copy()
df = df.sort_values([GROUP_COL, "timestamp"]).reset_index(drop=True)

print(f"Daylight rows: {len(df):,} / {rows_before_daylight:,}")
print(f"Maximum retained zenith: {df['zenith'].max():.4f}°")


## Derived features


In [ ]:
weather_variation = df.groupby("timestamp")[WEATHER_FEATURES].nunique(dropna=False)
assert not weather_variation.gt(1).any().any(),             "Weather columns vary by station at the same timestamp"

daylight_times = pd.DatetimeIndex(df["timestamp"].unique()).sort_values()
clearsky = location.get_clearsky(daylight_times, model="ineichen")
clearsky_by_time = pd.DataFrame({
    "timestamp": daylight_times,
    "clearsky_ghi": clearsky["ghi"].to_numpy(),
})
df = df.merge(clearsky_by_time, on="timestamp", how="left", validate="many_to_one")

df["clearsky_index"] = pvlib.irradiance.clearsky_index(
    df[GHI_COL], df["clearsky_ghi"], max_clearsky_index=2.0
)
df["cell_temp_ghi"] = pvlib.temperature.faiman(
    poa_global=df[GHI_COL],
    temp_air=df[TEMP_COL],
    wind_speed=df[WIND_COL],
    u0=25.0,
    u1=6.84,
)
df[TARGET_COL] = df[POWER_COL] / df["total_capacity_W"]

feature_ranges = df[
    ["clearsky_ghi", "clearsky_index", "cell_temp_ghi", TARGET_COL]
].agg(["min", "median", "max"]).T
feature_ranges


## Quality control


In [ ]:
def repeated_run_flag(frame, value_col, group_col=None):
    same_value = frame[value_col].eq(frame[value_col].shift())
    same_interval = frame["timestamp"].diff().eq(EXPECTED_INTERVAL)
    same_group = (
        frame[group_col].eq(frame[group_col].shift())
        if group_col else pd.Series(True, index=frame.index)
    )
    run_id = (~(same_value & same_interval & same_group)).cumsum()
    run_length = frame.groupby(run_id, sort=False)[value_col].transform("size")
    return run_length.ge(STUCK_MIN_SAMPLES)

df = df.sort_values([GROUP_COL, "timestamp"]).reset_index(drop=True)
df["flag_power_stuck"] = (
    repeated_run_flag(df, POWER_COL, GROUP_COL) & df[POWER_COL].gt(0)
)

weather = (
    df.drop_duplicates("timestamp")[["timestamp"] + WEATHER_FEATURES]
      .sort_values("timestamp").reset_index(drop=True)
)
weather["flag_weather_stuck"] = False
for column in WEATHER_FEATURES:
    weather["flag_weather_stuck"] |= repeated_run_flag(weather, column)

df = df.merge(
    weather[["timestamp", "flag_weather_stuck"]],
    on="timestamp", how="left", validate="many_to_one",
)
df["flag_fault_zero"] = df[POWER_COL].eq(0) & df[GHI_COL].ge(FAULT_ZERO_GHI)
df["flag_missing"] = df[MODEL_COLUMNS].isna().any(axis=1)
df["flag_impossible"] = (
    df["total_capacity_W"].le(0)
    | df[GHI_COL].lt(0)
    | df[WIND_COL].lt(0)
    | (df[RH_COL].notna() & ~df[RH_COL].between(0, 100))
    | (df[TARGET_COL].notna() & ~df[TARGET_COL].between(-0.001, 1.2))
)

MODEL_EXCLUSION_FLAGS = [
    "flag_missing", "flag_impossible", "flag_fault_zero",
    "flag_power_stuck", "flag_weather_stuck",
]
df["modeling_ready"] = ~df[MODEL_EXCLUSION_FLAGS].any(axis=1)

qc_flags = ["flag_after_gap"] + MODEL_EXCLUSION_FLAGS
qc_summary = pd.DataFrame({
    "rows": df[qc_flags].sum().astype("int64"),
    "percent": df[qc_flags].mean().mul(100).round(3),
})
qc_summary


## Validation


In [ ]:
ready = df["modeling_ready"]

assert df[GROUP_COL].nunique() == 37, "System count changed"
assert df["timestamp"].dt.tz is not None, "Timestamp lost its timezone"
assert df["zenith"].lt(90).all(), "Nighttime row survived"
assert df["clearsky_ghi"].gt(0).all(), "Non-positive clear-sky GHI found"
assert not df.duplicated([GROUP_COL, "timestamp"]).any(),             "Duplicate (station, timestamp) rows found"
assert df.loc[ready, MODEL_COLUMNS].notna().all().all(),             "Model-ready rows contain missing values"
assert not df.loc[ready, MODEL_EXCLUSION_FLAGS].any().any(),             "Model-ready rows contain an exclusion flag"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False, compression="zstd")

saved = pd.read_parquet(
    OUTPUT_PATH, columns=[GROUP_COL, "timestamp", "modeling_ready"]
)
assert len(saved) == len(df), "Saved Parquet row count changed"
assert saved[GROUP_COL].nunique() == 37, "Saved Parquet system count changed"
assert saved["timestamp"].dt.tz is not None, "Saved timestamp lost timezone"

phase1_summary = pd.Series({
    "systems": df[GROUP_COL].nunique(),
    "daylight_rows": len(df),
    "modeling_ready_rows": int(ready.sum()),
    "flagged_rows": int((~ready).sum()),
    "output_size_MB": round(OUTPUT_PATH.stat().st_size / 1_000_000, 1),
}, name="Phase 1 complete")
phase1_summary


# Phase 2 — Evaluation pipeline

Define metrics and leave-systems-out cross-validation.


In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def normalized_regression_metrics(y_true, y_pred):
    """Return metrics in fractions of installed capacity for y_norm.

    MBE uses prediction minus observation, so positive values indicate
    systematic overprediction.
    """
    observed = np.asarray(y_true, dtype=float).reshape(-1)
    predicted = np.asarray(y_pred, dtype=float).reshape(-1)
    if observed.shape != predicted.shape or observed.size == 0:
        raise ValueError("Observed and predicted values must be non-empty and aligned")
    if not (np.isfinite(observed).all() and np.isfinite(predicted).all()):
        raise ValueError("Metrics require finite observed and predicted values")

    error = predicted - observed
    return {
        "nRMSE": float(np.sqrt(np.mean(np.square(error)))),
        "MAE": float(np.mean(np.abs(error))),
        "MBE": float(np.mean(error)),
    }


## Cross-validation evaluator


In [ ]:
def evaluate_leave_systems_out(
    data,
    estimator,
    feature_columns,
    *,
    target_column=TARGET_COL,
    group_column=GROUP_COL,
    n_splits=5,
    scale_features=True,
    random_state=42,
):
    """Evaluate a regressor on unseen systems with leakage guards."""
    required = list(feature_columns) + [target_column, group_column, "modeling_ready"]
    missing = sorted(set(required) - set(data.columns))
    if missing:
        raise KeyError(f"Missing evaluation columns: {missing}")
    if target_column != TARGET_COL:
        raise ValueError(f"nRMSE is defined here for the capacity-normalized target {TARGET_COL!r}")

    evaluation = data.loc[data["modeling_ready"], required].copy()
    if evaluation[required].isna().any().any():
        raise ValueError("Model-ready evaluation data contains missing values")
    n_systems = evaluation[group_column].nunique()
    if not 2 <= n_splits <= n_systems:
        raise ValueError(f"n_splits must be between 2 and {n_systems}")

    X = evaluation.loc[:, feature_columns]
    y = evaluation[target_column]
    groups = evaluation[group_column]
    splitter = GroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_rows = []
    system_rows = []
    fitted_models = []
    tested_systems = []

    for fold, (train_index, test_index) in enumerate(
        splitter.split(X, y, groups=groups), start=1
    ):
        train_systems = set(groups.iloc[train_index])
        test_systems = set(groups.iloc[test_index])
        assert train_systems.isdisjoint(test_systems), (
            f"Fold {fold}: a test system leaked into training"
        )

        steps = []
        if scale_features:
            steps.append(("scaler", StandardScaler()))
        steps.append(("regressor", clone(estimator)))
        fitted = Pipeline(steps).fit(X.iloc[train_index], y.iloc[train_index])

        if scale_features:
            scaler = fitted.named_steps["scaler"]
            np.testing.assert_allclose(
                scaler.mean_,
                X.iloc[train_index].mean().to_numpy(),
                rtol=1e-10,
                atol=1e-12,
                err_msg=f"Fold {fold}: scaler was not fitted on training data only",
            )
            scaler_counts = np.asarray(scaler.n_samples_seen_)
            assert np.all(scaler_counts == len(train_index)), (
                f"Fold {fold}: scaler row count differs from training fold"
            )

        predictions = fitted.predict(X.iloc[test_index])
        fold_metrics = normalized_regression_metrics(y.iloc[test_index], predictions)
        fold_rows.append({
            "fold": fold,
            "n_train_rows": len(train_index),
            "n_test_rows": len(test_index),
            "n_train_systems": len(train_systems),
            "n_test_systems": len(test_systems),
            "observation_mean": float(y.iloc[test_index].mean()),
            "prediction_mean": float(np.mean(predictions)),
            "prediction_min": float(np.min(predictions)),
            "prediction_max": float(np.max(predictions)),
            **fold_metrics,
        })

        test_groups = groups.iloc[test_index].reset_index(drop=True)
        test_observed = y.iloc[test_index].reset_index(drop=True)
        predictions = np.asarray(predictions)
        for system in sorted(test_systems):
            system_mask = test_groups.eq(system).to_numpy()
            system_rows.append({
                "fold": fold,
                group_column: system,
                "n_test_rows": int(system_mask.sum()),
                **normalized_regression_metrics(
                    test_observed.to_numpy()[system_mask], predictions[system_mask]
                ),
            })
            tested_systems.append(system)
        fitted_models.append(fitted)

    expected_systems = set(evaluation[group_column].unique())
    assert set(tested_systems) == expected_systems, "Not every system was evaluated"
    assert len(tested_systems) == len(expected_systems), (
        "A system appeared in more than one test fold"
    )

    return {
        "fold_results": pd.DataFrame(fold_rows),
        "system_results": pd.DataFrame(system_rows).sort_values(group_column).reset_index(drop=True),
        "models": fitted_models,
    }


## Smoke test


In [ ]:
phase2_smoke_test = evaluate_leave_systems_out(
    df,
    DummyRegressor(strategy="mean"),
    WEATHER_FEATURES,
    n_splits=5,
    scale_features=True,
    random_state=42,
)

fold_results = phase2_smoke_test["fold_results"]
system_results = phase2_smoke_test["system_results"]
assert len(fold_results) == 5
assert len(system_results) == df.loc[df["modeling_ready"], GROUP_COL].nunique() == 37
assert system_results[GROUP_COL].is_unique
assert system_results[["nRMSE", "MAE", "MBE"]].notna().all().all()

system_metric_summary = (
    system_results[["nRMSE", "MAE", "MBE"]]
    .agg(["mean", "std", "min", "max"])
    .T
    .rename_axis("metric")
)
print("Leakage guards passed: 37 systems were each held out exactly once.")
display(fold_results.round(4))
display(system_metric_summary.round(4))
display(system_results.head(10).round(4))


# Phase 3 — Baselines

Evaluate the training-mean and GHI baselines.


In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_is_fitted

class GHIPhysicsRegressor(BaseEstimator, RegressorMixin):
    """Capacity-normalized GHI baseline with one training-only scale."""

    required_features = ("clearsky_ghi", "clearsky_index")

    def __init__(self, prediction_ceiling=1.2):
        self.prediction_ceiling = prediction_ceiling

    def _solar_signal(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("GHIPhysicsRegressor requires a pandas DataFrame")
        missing = sorted(set(self.required_features) - set(X.columns))
        if missing:
            raise KeyError(f"Missing GHI physics features: {missing}")

        feature_values = X.loc[:, self.required_features].to_numpy(dtype=float)
        if not np.isfinite(feature_values).all():
            raise ValueError("GHI physics features must be finite")
        if (feature_values < 0).any():
            raise ValueError("GHI physics features must be nonnegative")

        return (feature_values[:, 0] / 1000.0) * feature_values[:, 1]

    def fit(self, X, y):
        if not np.isfinite(self.prediction_ceiling) or self.prediction_ceiling <= 0:
            raise ValueError("prediction_ceiling must be finite and positive")

        signal = self._solar_signal(X)
        target = np.asarray(y, dtype=float).reshape(-1)
        if signal.shape != target.shape or target.size == 0:
            raise ValueError("Training features and target must be non-empty and aligned")
        if not np.isfinite(target).all():
            raise ValueError("Training target must be finite")

        signal_energy = float(np.dot(signal, signal))
        if signal_energy <= 0:
            raise ValueError("Training fold has no positive GHI physics signal")

        self.scale_ = max(0.0, float(np.dot(signal, target) / signal_energy))
        self.n_features_in_ = X.shape[1]
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def predict(self, X):
        check_is_fitted(self, attributes=["scale_"])
        prediction = self.scale_ * self._solar_signal(X)
        return np.clip(prediction, 0.0, self.prediction_ceiling)


## Run baselines


In [ ]:
PHASE3_N_SPLITS = 5
PHASE3_RANDOM_STATE = 42
PHASE3_METRICS = ["nRMSE", "MAE", "MBE"]
GHI_BASELINE_FEATURES = ["clearsky_ghi", "clearsky_index"]

training_mean_baseline = evaluate_leave_systems_out(
    df,
    DummyRegressor(strategy="mean"),
    [GHI_COL],  # Required by the estimator API but ignored by DummyRegressor.
    n_splits=PHASE3_N_SPLITS,
    scale_features=False,
    random_state=PHASE3_RANDOM_STATE,
)

ghi_physics_baseline = evaluate_leave_systems_out(
    df,
    GHIPhysicsRegressor(prediction_ceiling=1.2),
    GHI_BASELINE_FEATURES,
    n_splits=PHASE3_N_SPLITS,
    scale_features=False,
    random_state=PHASE3_RANDOM_STATE,
)

phase3_runs = {
    "training_mean": training_mean_baseline,
    "ghi_physics": ghi_physics_baseline,
}
phase3_system_results = pd.concat(
    [result["system_results"].assign(model=name) for name, result in phase3_runs.items()],
    ignore_index=True,
).loc[:, ["model", "fold", GROUP_COL, "n_test_rows", *PHASE3_METRICS]]

phase3_fold_results = pd.concat(
    [result["fold_results"].assign(model=name) for name, result in phase3_runs.items()],
    ignore_index=True,
)
phase3_fold_results = phase3_fold_results.loc[
    :, ["model", *[column for column in phase3_fold_results.columns if column != "model"]]
]

phase3_physics_parameters = pd.DataFrame([
    {
        "model": "ghi_physics",
        "fold": fold,
        "scale": fitted.named_steps["regressor"].scale_,
        "prediction_ceiling": fitted.named_steps["regressor"].prediction_ceiling,
    }
    for fold, fitted in enumerate(ghi_physics_baseline["models"], start=1)
])


## Validation


In [ ]:
expected_systems = set(df.loc[df["modeling_ready"], GROUP_COL].unique())
assert len(expected_systems) == 37
assert len(phase3_system_results) == 2 * len(expected_systems)
assert phase3_system_results.groupby("model")[GROUP_COL].nunique().eq(37).all()
assert phase3_system_results.groupby(["model", GROUP_COL]).size().eq(1).all()
assert np.isfinite(phase3_system_results[PHASE3_METRICS].to_numpy()).all()

fold_assignment = phase3_system_results.pivot(
    index=GROUP_COL, columns="model", values="fold"
)
assert set(fold_assignment.index) == expected_systems
assert fold_assignment["training_mean"].eq(fold_assignment["ghi_physics"]).all(), (
    "Baselines did not use identical held-out-system folds"
)

assert np.isfinite(phase3_physics_parameters["scale"]).all()
assert phase3_physics_parameters["scale"].ge(0).all()
assert phase3_fold_results["prediction_min"].ge(-1e-12).all()
assert phase3_fold_results["prediction_max"].le(1.2 + 1e-12).all()

ghi_physics_repeat = evaluate_leave_systems_out(
    df,
    GHIPhysicsRegressor(prediction_ceiling=1.2),
    GHI_BASELINE_FEATURES,
    n_splits=PHASE3_N_SPLITS,
    scale_features=False,
    random_state=PHASE3_RANDOM_STATE,
)
pd.testing.assert_frame_equal(
    ghi_physics_baseline["fold_results"],
    ghi_physics_repeat["fold_results"],
)
pd.testing.assert_frame_equal(
    ghi_physics_baseline["system_results"],
    ghi_physics_repeat["system_results"],
)
print("Phase 3 checks passed: both baselines evaluated the same 37 systems deterministically.")


## Results


In [ ]:
phase3_summary = (
    phase3_system_results.groupby("model")[PHASE3_METRICS]
    .agg(["mean", "std", "median", "min", "max"])
)
phase3_summary.columns = ["_".join(column) for column in phase3_summary.columns]
phase3_summary = phase3_summary.reset_index()
system_counts = phase3_system_results.groupby("model")[GROUP_COL].nunique()
phase3_summary.insert(1, "n_systems", phase3_summary["model"].map(system_counts))

comparison_rows = []
for metric in PHASE3_METRICS:
    paired_metric = phase3_system_results.pivot(
        index=GROUP_COL, columns="model", values=metric
    )
    mean_reference = float(paired_metric["training_mean"].mean())
    mean_physics = float(paired_metric["ghi_physics"].mean())
    comparison_rows.append({
        "metric": metric,
        "training_mean": mean_reference,
        "ghi_physics": mean_physics,
        "physics_minus_mean": mean_physics - mean_reference,
        "relative_reduction_pct": (
            100.0 * (mean_reference - mean_physics) / mean_reference
            if metric in {"nRMSE", "MAE"} else np.nan
        ),
    })
phase3_comparison = pd.DataFrame(comparison_rows)

nrmse_pairs = phase3_system_results.pivot(
    index=GROUP_COL, columns="model", values="nRMSE"
)
physics_wins = int(nrmse_pairs["ghi_physics"].lt(nrmse_pairs["training_mean"]).sum())

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
phase3_system_results.to_csv(RESULTS_DIR / "phase3_system_results.csv", index=False)
phase3_fold_results.to_csv(RESULTS_DIR / "phase3_fold_results.csv", index=False)
phase3_summary.to_csv(RESULTS_DIR / "phase3_summary.csv", index=False)
phase3_physics_parameters.to_csv(
    RESULTS_DIR / "phase3_physics_parameters.csv", index=False
)

print(f"GHI physics has lower nRMSE on {physics_wins} of {len(nrmse_pairs)} systems.")
display(phase3_summary.round(4))
display(phase3_comparison.round(4))
display(phase3_physics_parameters.round(4))


In [ ]:
import matplotlib.pyplot as plt

plot_min = float(nrmse_pairs.min().min())
plot_max = float(nrmse_pairs.max().max())
plot_padding = 0.05 * (plot_max - plot_min)
plot_limits = (plot_min - plot_padding, plot_max + plot_padding)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    nrmse_pairs["training_mean"],
    nrmse_pairs["ghi_physics"],
    alpha=0.8,
    edgecolor="white",
    linewidth=0.5,
)
ax.plot(plot_limits, plot_limits, linestyle="--", color="black", linewidth=1)
ax.set(
    xlim=plot_limits,
    ylim=plot_limits,
    xlabel="Training-mean nRMSE",
    ylabel="GHI-physics nRMSE",
    title="Held-out-system error: GHI physics vs. training mean",
)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


# Phase 4 — Models A and B

Compare weather-only XGBoost with weather + physics features.


In [ ]:
import importlib
import subprocess
import sys

try:
    from xgboost import XGBRegressor
except ModuleNotFoundError as error:
    if error.name != "xgboost":
        raise
    print(f"Installing XGBoost into {sys.executable} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    importlib.invalidate_caches()
    from xgboost import XGBRegressor

MODEL_A_FEATURES = list(WEATHER_FEATURES)
MODEL_B_FEATURES = list(WEATHER_FEATURES) + list(PHYSICS_FEATURES)

PHASE4_N_SPLITS = 5
PHASE4_RANDOM_STATE = 42
PHASE4_METRICS = ["nRMSE", "MAE", "MBE"]
PHASE4_MODEL_CONFIG = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "random_state": PHASE4_RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 1,
}

def make_phase4_estimator():
    """Return a fresh XGBoost regressor with the shared Phase 4 settings."""
    return XGBRegressor(**PHASE4_MODEL_CONFIG)

assert MODEL_A_FEATURES == WEATHER_FEATURES
assert MODEL_B_FEATURES[:len(MODEL_A_FEATURES)] == MODEL_A_FEATURES
assert MODEL_B_FEATURES[len(MODEL_A_FEATURES):] == PHYSICS_FEATURES
assert len(MODEL_B_FEATURES) == len(set(MODEL_B_FEATURES))
assert set(MODEL_B_FEATURES).issubset(df.columns)
assert "poa_global" not in MODEL_B_FEATURES

phase4_design = pd.DataFrame({
    "model": ["model_a_weather", "model_b_hybrid"],
    "n_features": [len(MODEL_A_FEATURES), len(MODEL_B_FEATURES)],
    "features": [MODEL_A_FEATURES, MODEL_B_FEATURES],
})
print("XGBoost available in this kernel: True")
display(phase4_design)


## Train models


In [ ]:
model_a_weather = evaluate_leave_systems_out(
    df,
    make_phase4_estimator(),
    MODEL_A_FEATURES,
    n_splits=PHASE4_N_SPLITS,
    scale_features=False,
    random_state=PHASE4_RANDOM_STATE,
)
model_b_hybrid = evaluate_leave_systems_out(
    df,
    make_phase4_estimator(),
    MODEL_B_FEATURES,
    n_splits=PHASE4_N_SPLITS,
    scale_features=False,
    random_state=PHASE4_RANDOM_STATE,
)
phase4_runs = {
    "model_a_weather": model_a_weather,
    "model_b_hybrid": model_b_hybrid,
}
print("Phase 4 training complete.")


## Results


In [ ]:
phase4_system_results = pd.concat(
    [
        run["system_results"].assign(model=model_name)
        for model_name, run in phase4_runs.items()
    ],
    ignore_index=True,
).loc[:, ["model", "fold", GROUP_COL, "n_test_rows", *PHASE4_METRICS]]

phase4_fold_results = pd.concat(
    [
        run["fold_results"].assign(model=model_name)
        for model_name, run in phase4_runs.items()
    ],
    ignore_index=True,
)
phase4_fold_results = phase4_fold_results.loc[
    :, ["model", *[c for c in phase4_fold_results.columns if c != "model"]]
]

fold_assignments = phase4_system_results.pivot(
    index=GROUP_COL, columns="model", values="fold"
)
assert fold_assignments.notna().all().all()
assert fold_assignments["model_a_weather"].equals(
    fold_assignments["model_b_hybrid"]
), "Models A and B did not use identical held-out-system folds"
assert phase4_system_results.groupby("model")[GROUP_COL].nunique().eq(37).all()
assert np.isfinite(phase4_system_results[PHASE4_METRICS]).all().all()

phase4_summary = (
    phase4_system_results.groupby("model")[PHASE4_METRICS]
    .agg(["mean", "std", "median", "min", "max"])
)
phase4_summary.columns = ["_".join(column) for column in phase4_summary.columns]
phase4_summary = phase4_summary.reset_index()

paired_metrics = phase4_system_results.pivot(
    index=GROUP_COL, columns="model", values=PHASE4_METRICS
)
phase4_paired_comparison = pd.DataFrame([
    {
        "metric": metric,
        "model_a_weather": paired_metrics[(metric, "model_a_weather")].mean(),
        "model_b_hybrid": paired_metrics[(metric, "model_b_hybrid")].mean(),
        "hybrid_minus_weather": (
            paired_metrics[(metric, "model_b_hybrid")].mean()
            - paired_metrics[(metric, "model_a_weather")].mean()
        ),
        "relative_error_reduction_pct": (
            100.0
            * (
                paired_metrics[(metric, "model_a_weather")].mean()
                - paired_metrics[(metric, "model_b_hybrid")].mean()
            )
            / paired_metrics[(metric, "model_a_weather")].mean()
            if metric in {"nRMSE", "MAE"}
            else np.nan
        ),
    }
    for metric in PHASE4_METRICS
])

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
phase4_system_results.to_csv(
    RESULTS_DIR / "phase4_system_results.csv", index=False
)
phase4_fold_results.to_csv(
    RESULTS_DIR / "phase4_fold_results.csv", index=False
)
phase4_summary.to_csv(RESULTS_DIR / "phase4_summary.csv", index=False)
phase4_paired_comparison.to_csv(
    RESULTS_DIR / "phase4_paired_comparison.csv", index=False
)

print("Phase 4 checks passed: Models A and B used identical held-out-system folds.")
display(phase4_summary.round(4))
display(phase4_paired_comparison.round(4))


# Phase 5 — Combined evaluation

Combine the Phase 3 and Phase 4 results and compare all four models.


In [ ]:
PHASE5_METRICS = ["nRMSE", "MAE", "MBE"]
PHASE5_ERROR_METRICS = ["nRMSE", "MAE"]
PHASE5_MODEL_ORDER = [
    "training_mean",
    "ghi_physics",
    "model_a_weather",
    "model_b_hybrid",
]
PHASE5_MODEL_LABELS = {
    "training_mean": "Training mean",
    "ghi_physics": "GHI physics",
    "model_a_weather": "Model A: weather",
    "model_b_hybrid": "Model B: hybrid",
}

PHASE5_N_SPLITS = 5
PHASE5_RANDOM_STATE = 42
PHASE5_BOOTSTRAP_SAMPLES = 10_000

def validate_phase5_tables(system_results, fold_results, *, expected_systems=None):
    """Reject unpaired or incomplete four-model comparisons."""
    system_required = {
        "model", "fold", GROUP_COL, "n_test_rows", *PHASE5_METRICS
    }
    fold_required = {
        "model", "fold", "n_train_rows", "n_test_rows",
        "n_train_systems", "n_test_systems", "observation_mean",
        *PHASE5_METRICS,
    }
    missing_system = sorted(system_required - set(system_results))
    missing_fold = sorted(fold_required - set(fold_results))
    if missing_system or missing_fold:
        raise KeyError(
            f"Missing system columns {missing_system}; missing fold columns {missing_fold}"
        )

    observed_models = set(system_results["model"])
    if observed_models != set(PHASE5_MODEL_ORDER):
        raise ValueError(
            f"Expected models {PHASE5_MODEL_ORDER}, found {sorted(observed_models)}"
        )
    if system_results.duplicated(["model", GROUP_COL]).any():
        raise ValueError("A model has duplicate system-level result rows")
    if fold_results.duplicated(["model", "fold"]).any():
        raise ValueError("A model has duplicate fold-level result rows")
    if not np.isfinite(system_results[PHASE5_METRICS]).all().all():
        raise ValueError("System-level metrics contain non-finite values")
    if not np.isfinite(fold_results[PHASE5_METRICS]).all().all():
        raise ValueError("Fold-level metrics contain non-finite values")
    if system_results[PHASE5_ERROR_METRICS].lt(0).any().any():
        raise ValueError("nRMSE and MAE cannot be negative")

    systems_by_model = system_results.groupby("model")[GROUP_COL].agg(set)
    reference_systems = systems_by_model.loc[PHASE5_MODEL_ORDER[0]]
    if not systems_by_model.map(lambda systems: systems == reference_systems).all():
        raise ValueError("The four models were not evaluated on identical systems")
    if expected_systems is not None and reference_systems != set(expected_systems):
        raise ValueError("Phase 5 system coverage differs from the prepared dataset")

    fold_assignment = system_results.pivot(
        index=GROUP_COL, columns="model", values="fold"
    ).loc[:, PHASE5_MODEL_ORDER]
    row_counts = system_results.pivot(
        index=GROUP_COL, columns="model", values="n_test_rows"
    ).loc[:, PHASE5_MODEL_ORDER]
    if fold_assignment.isna().any().any() or row_counts.isna().any().any():
        raise ValueError("A model is missing a held-out system result")
    if not fold_assignment.eq(fold_assignment.iloc[:, 0], axis=0).all().all():
        raise ValueError("Models used different held-out folds for a system")
    if not row_counts.eq(row_counts.iloc[:, 0], axis=0).all().all():
        raise ValueError("Models used different test-row counts for a system")

    fold_models = set(fold_results["model"])
    if fold_models != set(PHASE5_MODEL_ORDER):
        raise ValueError("Fold-level results do not contain all four models")
    fold_sets = fold_results.groupby("model")["fold"].agg(set)
    reference_folds = fold_sets.loc[PHASE5_MODEL_ORDER[0]]
    if not fold_sets.map(lambda folds: folds == reference_folds).all():
        raise ValueError("Models do not contain identical fold IDs")

    fold_comparison_columns = [
        "n_train_rows", "n_test_rows", "n_train_systems",
        "n_test_systems", "observation_mean",
    ]
    for column in fold_comparison_columns:
        comparison = fold_results.pivot(
            index="fold", columns="model", values=column
        ).loc[:, PHASE5_MODEL_ORDER]
        reference = comparison.iloc[:, 0].to_numpy(dtype=float)
        if not all(
            np.allclose(
                comparison[model].to_numpy(dtype=float),
                reference,
                rtol=1e-10,
                atol=1e-12,
            )
            for model in PHASE5_MODEL_ORDER[1:]
        ):
            raise ValueError(f"Fold composition differs across models for {column}")

    return {
        "n_models": len(PHASE5_MODEL_ORDER),
        "n_systems": len(reference_systems),
        "n_folds": len(reference_folds),
    }

def paired_bootstrap_summary(
    system_results,
    metric,
    *,
    reference_model="model_a_weather",
    candidate_model="model_b_hybrid",
    n_bootstrap=PHASE5_BOOTSTRAP_SAMPLES,
    random_state=PHASE5_RANDOM_STATE,
):
    """Summarize a paired candidate-minus-reference system comparison."""
    paired = system_results.pivot(
        index=GROUP_COL, columns="model", values=metric
    ).loc[:, [reference_model, candidate_model]].dropna()
    reference = paired[reference_model].to_numpy(dtype=float)
    candidate = paired[candidate_model].to_numpy(dtype=float)
    difference = candidate - reference

    rng = np.random.default_rng(random_state)
    bootstrap_indices = rng.integers(
        0, len(paired), size=(n_bootstrap, len(paired))
    )
    bootstrap_reference = reference[bootstrap_indices].mean(axis=1)
    bootstrap_candidate = candidate[bootstrap_indices].mean(axis=1)
    bootstrap_difference = bootstrap_candidate - bootstrap_reference

    row = {
        "metric": metric,
        "reference_model": reference_model,
        "candidate_model": candidate_model,
        "n_systems": len(paired),
        "reference_macro_mean": reference.mean(),
        "candidate_macro_mean": candidate.mean(),
        "candidate_minus_reference": difference.mean(),
        "difference_ci95_low": np.quantile(bootstrap_difference, 0.025),
        "difference_ci95_high": np.quantile(bootstrap_difference, 0.975),
        "candidate_better_systems": int(
            (candidate < reference).sum()
            if metric in PHASE5_ERROR_METRICS
            else (np.abs(candidate) < np.abs(reference)).sum()
        ),
        "ties": int(np.isclose(candidate, reference, rtol=0.0, atol=1e-12).sum()),
    }

    if metric in PHASE5_ERROR_METRICS:
        bootstrap_reduction = (
            100.0
            * (bootstrap_reference - bootstrap_candidate)
            / bootstrap_reference
        )
        row.update({
            "relative_error_reduction_pct": (
                100.0 * (reference.mean() - candidate.mean()) / reference.mean()
            ),
            "relative_reduction_ci95_low": np.quantile(
                bootstrap_reduction, 0.025
            ),
            "relative_reduction_ci95_high": np.quantile(
                bootstrap_reduction, 0.975
            ),
        })
    else:
        row.update({
            "relative_error_reduction_pct": np.nan,
            "relative_reduction_ci95_low": np.nan,
            "relative_reduction_ci95_high": np.nan,
        })
    return row


In [ ]:
phase5_system_results = pd.concat(
    [phase3_system_results.copy(), phase4_system_results.copy()],
    ignore_index=True,
)
phase5_fold_results = pd.concat(
    [phase3_fold_results.copy(), phase4_fold_results.copy()],
    ignore_index=True,
)

expected_phase5_systems = set(
    df.loc[df["modeling_ready"], GROUP_COL].unique()
)
phase5_validation = validate_phase5_tables(
    phase5_system_results,
    phase5_fold_results,
    expected_systems=expected_phase5_systems,
)
print(
    "Phase 5 inputs validated: "
    f"{phase5_validation['n_models']} models, "
    f"{phase5_validation['n_systems']} systems, "
    f"{phase5_validation['n_folds']} paired folds."
)


## Results


In [ ]:
phase5_summary = (
    phase5_system_results.groupby("model")[PHASE5_METRICS]
    .agg(["mean", "std", "median"])
)
phase5_summary.columns = [
    f"{metric}_{statistic}" for metric, statistic in phase5_summary.columns
]
phase5_summary = phase5_summary.reset_index()
phase5_summary.insert(
    1,
    "label",
    phase5_summary["model"].map(PHASE5_MODEL_LABELS),
)
phase5_summary.insert(
    2,
    "n_systems",
    phase5_summary["model"].map(
        phase5_system_results.groupby("model")[GROUP_COL].nunique()
    ),
)
phase5_summary["model"] = pd.Categorical(
    phase5_summary["model"], categories=PHASE5_MODEL_ORDER, ordered=True
)
phase5_summary = phase5_summary.sort_values("model").reset_index(drop=True)
phase5_summary["model"] = phase5_summary["model"].astype(str)

phase5_a_vs_b = pd.DataFrame([
    paired_bootstrap_summary(phase5_system_results, metric)
    for metric in PHASE5_METRICS
])

nrmse_headline = phase5_a_vs_b.loc[
    phase5_a_vs_b["metric"].eq("nRMSE")
].iloc[0]
phase5_headline = pd.DataFrame([{
    "n_systems": int(nrmse_headline["n_systems"]),
    "model_a_macro_nRMSE": nrmse_headline["reference_macro_mean"],
    "model_b_macro_nRMSE": nrmse_headline["candidate_macro_mean"],
    "hybrid_minus_weather_nRMSE": nrmse_headline[
        "candidate_minus_reference"
    ],
    "relative_nRMSE_reduction_pct": nrmse_headline[
        "relative_error_reduction_pct"
    ],
    "relative_reduction_ci95_low": nrmse_headline[
        "relative_reduction_ci95_low"
    ],
    "relative_reduction_ci95_high": nrmse_headline[
        "relative_reduction_ci95_high"
    ],
    "hybrid_better_systems": int(nrmse_headline["candidate_better_systems"]),
}])

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
phase5_system_results.to_csv(
    RESULTS_DIR / "phase5_system_results.csv", index=False
)
phase5_fold_results.to_csv(
    RESULTS_DIR / "phase5_fold_results.csv", index=False
)
phase5_summary.to_csv(RESULTS_DIR / "phase5_summary.csv", index=False)
phase5_a_vs_b.to_csv(
    RESULTS_DIR / "phase5_a_vs_b_comparison.csv", index=False
)
phase5_headline.to_csv(RESULTS_DIR / "phase5_headline.csv", index=False)

print(
    f"Model B macro nRMSE: {nrmse_headline['candidate_macro_mean']:.4f}; "
    f"Model A: {nrmse_headline['reference_macro_mean']:.4f}; "
    f"relative reduction: "
    f"{nrmse_headline['relative_error_reduction_pct']:.1f}% "
    f"(paired bootstrap 95% CI "
    f"{nrmse_headline['relative_reduction_ci95_low']:.1f}% to "
    f"{nrmse_headline['relative_reduction_ci95_high']:.1f}%)."
)
display(phase5_summary.round(4))
display(phase5_a_vs_b.round(4))


## Plots


In [ ]:
phase5_plot_summary = phase5_summary.loc[
    :, ["model", "label", "n_systems", "nRMSE_mean"]
].copy()
phase5_paired_nrmse = phase5_system_results.pivot(
    index=GROUP_COL, columns="model", values="nRMSE"
).reset_index()
phase5_paired_nrmse = phase5_paired_nrmse.merge(
    phase5_system_results.loc[
        phase5_system_results["model"].eq("model_a_weather"),
        [GROUP_COL, "n_test_rows"],
    ],
    on=GROUP_COL,
    how="left",
    validate="one_to_one",
)
phase5_paired_nrmse["hybrid_lower_error"] = (
    phase5_paired_nrmse["model_b_hybrid"]
    < phase5_paired_nrmse["model_a_weather"]
)

bar_colors = ["#7A8088", "#D49A20", "#3478B8", "#D9693A"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))

bars = axes[0].barh(
    phase5_plot_summary["label"],
    phase5_plot_summary["nRMSE_mean"],
    color=bar_colors,
    edgecolor="#27313A",
    linewidth=0.8,
)
axes[0].invert_yaxis()
axes[0].set_xlim(
    0,
    phase5_plot_summary["nRMSE_mean"].max() * 1.18,
)
axes[0].set_xlabel("Macro nRMSE (fraction of installed capacity)")
axes[0].set_title(
    "Macro nRMSE by model\n"
    f"System-weighted mean across {phase5_validation['n_systems']} held-out systems",
    loc="left",
)
axes[0].bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
axes[0].grid(axis="x", color="#D9DEE3", linewidth=0.8)
axes[0].set_axisbelow(True)

improved = phase5_paired_nrmse["hybrid_lower_error"]
axes[1].scatter(
    phase5_paired_nrmse.loc[improved, "model_a_weather"],
    phase5_paired_nrmse.loc[improved, "model_b_hybrid"],
    s=48,
    color="#3478B8",
    edgecolor="#27313A",
    linewidth=0.6,
    label="Hybrid lower nRMSE",
)
axes[1].scatter(
    phase5_paired_nrmse.loc[~improved, "model_a_weather"],
    phase5_paired_nrmse.loc[~improved, "model_b_hybrid"],
    s=48,
    facecolor="white",
    edgecolor="#D9693A",
    linewidth=1.4,
    label="Weather-only lower or tied",
)
plot_limit = 1.06 * max(
    phase5_paired_nrmse["model_a_weather"].max(),
    phase5_paired_nrmse["model_b_hybrid"].max(),
)
axes[1].plot(
    [0, plot_limit], [0, plot_limit],
    color="#27313A", linestyle="--", linewidth=1.0,
    label="Equal error",
)
axes[1].set_xlim(0, plot_limit)
axes[1].set_ylim(0, plot_limit)
axes[1].set_aspect("equal", adjustable="box")
axes[1].set_xlabel("Model A system nRMSE")
axes[1].set_ylabel("Model B system nRMSE")
axes[1].set_title(
    "System-level nRMSE: Model A versus Model B\n"
    f"Each point is one held-out PV system (n={len(phase5_paired_nrmse)})",
    loc="left",
)
axes[1].grid(color="#E4E8EC", linewidth=0.7)
axes[1].set_axisbelow(True)
axes[1].legend(frameon=False, fontsize=8, loc="upper left")

for axis in axes:
    axis.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
phase5_figure_path = RESULTS_DIR / "phase5_main_comparison.png"
fig.savefig(phase5_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved {phase5_figure_path}")


# Phase 6 — Future-time Model D

Compare weather, time, physics, and their combined representation on unseen 2023 systems.


In [ ]:
PHASE6_TRAIN_END = pd.Timestamp("2022-12-31 23:59:59", tz=TIMEZONE)
PHASE6_TEST_START = pd.Timestamp("2023-01-01 00:00:00", tz=TIMEZONE)
PHASE6_N_SPLITS = 5
PHASE6_RANDOM_STATE = 42
PHASE6_N_JOBS = 4

assert df["timestamp"].dt.tz is not None
phase6_fractional_hour = (
    df["timestamp"].dt.hour
    + df["timestamp"].dt.minute / 60.0
    + df["timestamp"].dt.second / 3600.0
)
phase6_day_of_year = df["timestamp"].dt.dayofyear.astype(float) - 1.0
df["clock_hour_sin"] = np.sin(2.0 * np.pi * phase6_fractional_hour / 24.0)
df["clock_hour_cos"] = np.cos(2.0 * np.pi * phase6_fractional_hour / 24.0)
df["calendar_doy_sin"] = np.sin(
    2.0 * np.pi * phase6_day_of_year / 365.2425
)
df["calendar_doy_cos"] = np.cos(
    2.0 * np.pi * phase6_day_of_year / 365.2425
)

PHASE6_CLOCK_FEATURES = [
    "clock_hour_sin",
    "clock_hour_cos",
    "calendar_doy_sin",
    "calendar_doy_cos",
]

PHASE6_TIME_ONLY_FEATURES = ["zenith", "azimuth", "clearsky_ghi"]
PHASE6_WEATHER_FEATURES = list(WEATHER_FEATURES)
PHASE6_MODEL_C_FEATURES = (
    list(WEATHER_FEATURES) + list(PHASE6_CLOCK_FEATURES)
)
PHASE6_HYBRID_FEATURES = list(WEATHER_FEATURES) + list(PHYSICS_FEATURES)
PHASE6_MODEL_D_FEATURES = (
    list(WEATHER_FEATURES)
    + list(PHASE6_CLOCK_FEATURES)
    + list(PHYSICS_FEATURES)
)

PHASE6_MODEL_ORDER = [
    "training_mean",
    "time_solar_only",
    "model_a_weather",
    "model_c_weather_time",
    "model_b_hybrid",
    "model_d_weather_time_physics",
]
PHASE6_MODEL_LABELS = {
    "training_mean": "Training mean",
    "time_solar_only": "Time/solar only",
    "model_a_weather": "Model A: weather",
    "model_c_weather_time": "Model C: weather + time",
    "model_b_hybrid": "Model B: weather + physics",
    "model_d_weather_time_physics": "Model D: weather + time + physics",
}

assert PHASE6_TRAIN_END < PHASE6_TEST_START
assert set(PHASE6_TIME_ONLY_FEATURES).issubset(df.columns)
assert set(PHASE6_MODEL_C_FEATURES).issubset(df.columns)
assert set(PHASE6_HYBRID_FEATURES).issubset(df.columns)
assert set(PHASE6_MODEL_D_FEATURES).issubset(df.columns)
assert PHASE6_MODEL_C_FEATURES[:len(WEATHER_FEATURES)] == WEATHER_FEATURES
assert PHASE6_MODEL_C_FEATURES[len(WEATHER_FEATURES):] == PHASE6_CLOCK_FEATURES
assert df[PHASE6_CLOCK_FEATURES].notna().all().all()
assert len(PHASE6_MODEL_D_FEATURES) == len(set(PHASE6_MODEL_D_FEATURES))


## Split audit


In [ ]:
def build_phase6_station_folds(
    data,
    *,
    n_splits=PHASE6_N_SPLITS,
    random_state=PHASE6_RANDOM_STATE,
):
    """Assign every station to one deterministic held-out fold."""
    eligible = data.loc[data["modeling_ready"], [GROUP_COL]].copy()
    systems = eligible[GROUP_COL].drop_duplicates().sort_values().reset_index(drop=True)
    if not 2 <= n_splits <= len(systems):
        raise ValueError(f"n_splits must be between 2 and {len(systems)}")

    splitter = GroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )
    fold_by_station = {}
    placeholder = np.zeros((len(systems), 1), dtype=np.uint8)
    for fold, (_, test_index) in enumerate(
        splitter.split(placeholder, groups=systems), start=1
    ):
        for station in systems.iloc[test_index]:
            if station in fold_by_station:
                raise AssertionError(f"{station} was assigned more than once")
            fold_by_station[station] = fold

    assert set(fold_by_station) == set(systems)
    return fold_by_station

def audit_phase6_splits(
    data,
    fold_by_station,
    feature_columns,
    *,
    train_end=PHASE6_TRAIN_END,
    test_start=PHASE6_TEST_START,
):
    """Check system and time isolation without fitting any estimator."""
    required = {
        GROUP_COL, "timestamp", TARGET_COL, "modeling_ready", *feature_columns
    }
    missing = sorted(required - set(data.columns))
    if missing:
        raise KeyError(f"Missing Phase 6 columns: {missing}")

    ready = data.loc[data["modeling_ready"], list(required)].copy()
    if ready[list(required)].isna().any().any():
        raise ValueError("Phase 6 model-ready rows contain missing values")

    rows = []
    tested_systems = []
    for fold in sorted(set(fold_by_station.values())):
        test_systems = {
            station for station, assigned_fold in fold_by_station.items()
            if assigned_fold == fold
        }
        assigned_train_systems = set(fold_by_station) - test_systems

        train_mask = (
            ready[GROUP_COL].isin(assigned_train_systems)
            & ready["timestamp"].le(train_end)
        )
        test_mask = (
            ready[GROUP_COL].isin(test_systems)
            & ready["timestamp"].ge(test_start)
        )
        train_part = ready.loc[train_mask]
        test_part = ready.loc[test_mask]

        if train_part.empty or test_part.empty:
            raise ValueError(f"Fold {fold} has an empty train or test partition")
        actual_train_systems = set(train_part[GROUP_COL])
        actual_test_systems = set(test_part[GROUP_COL])
        assert assigned_train_systems.isdisjoint(test_systems)
        assert actual_train_systems.issubset(assigned_train_systems)
        assert actual_test_systems == test_systems
        assert set(train_part[GROUP_COL]).isdisjoint(set(test_part[GROUP_COL]))
        assert train_part["timestamp"].max() < test_part["timestamp"].min()
        assert set(train_part["timestamp"].unique()).isdisjoint(
            set(test_part["timestamp"].unique())
        )

        tested_systems.extend(sorted(test_systems))
        rows.append({
            "fold": fold,
            "n_assigned_train_systems": len(assigned_train_systems),
            "n_train_systems_with_past_rows": len(actual_train_systems),
            "n_test_systems": len(test_systems),
            "n_train_rows": len(train_part),
            "n_test_rows": len(test_part),
            "train_start": train_part["timestamp"].min(),
            "train_end": train_part["timestamp"].max(),
            "test_start": test_part["timestamp"].min(),
            "test_end": test_part["timestamp"].max(),
        })

    assert len(tested_systems) == len(set(tested_systems)) == len(fold_by_station)
    return pd.DataFrame(rows)

phase6_fold_by_station = build_phase6_station_folds(df)
phase6_station_coverage = (
    df.loc[df["modeling_ready"]]
    .groupby(GROUP_COL)["timestamp"]
    .agg(first_timestamp="min", last_timestamp="max", n_ready_rows="size")
    .reset_index()
)
phase6_no_past_systems = phase6_station_coverage.loc[
    phase6_station_coverage["first_timestamp"].gt(PHASE6_TRAIN_END),
    GROUP_COL,
].tolist()
phase6_split_audit = audit_phase6_splits(
    df,
    phase6_fold_by_station,
    PHASE6_MODEL_D_FEATURES,
)
print("Phase 6 split audit passed. No model was trained.")
print(
    "Stations with no pre-2023 training history (still eligible as 2023 "
    f"cold-start test systems): {phase6_no_past_systems}"
)
display(phase6_split_audit)


## Evaluator


In [ ]:
def make_phase6_estimator():
    """Use the Phase 4 model settings with a safer CPU-thread limit."""
    config = dict(PHASE4_MODEL_CONFIG)
    config["n_jobs"] = PHASE6_N_JOBS
    config["random_state"] = PHASE6_RANDOM_STATE
    return XGBRegressor(**config)

def evaluate_future_unseen_systems(
    data,
    estimator,
    feature_columns,
    fold_by_station,
    *,
    train_end=PHASE6_TRAIN_END,
    test_start=PHASE6_TEST_START,
):
    """Evaluate later timestamps from stations wholly absent from training."""
    required = [
        GROUP_COL, "timestamp", TARGET_COL, "modeling_ready", *feature_columns
    ]
    evaluation = data.loc[data["modeling_ready"], required].copy()
    if evaluation[required].isna().any().any():
        raise ValueError("Phase 6 evaluation data contains missing values")

    fold_rows = []
    system_rows = []
    tested_systems = []

    for fold in sorted(set(fold_by_station.values())):
        test_systems = {
            station for station, assigned_fold in fold_by_station.items()
            if assigned_fold == fold
        }
        assigned_train_systems = set(fold_by_station) - test_systems
        train_mask = (
            evaluation[GROUP_COL].isin(assigned_train_systems)
            & evaluation["timestamp"].le(train_end)
        )
        test_mask = (
            evaluation[GROUP_COL].isin(test_systems)
            & evaluation["timestamp"].ge(test_start)
        )
        train_part = evaluation.loc[train_mask]
        test_part = evaluation.loc[test_mask]

        if train_part.empty or test_part.empty:
            raise ValueError(f"Fold {fold} has an empty train or test partition")
        assert set(train_part[GROUP_COL]).isdisjoint(set(test_part[GROUP_COL]))
        actual_train_systems = set(train_part[GROUP_COL])
        assert actual_train_systems.issubset(assigned_train_systems)
        assert set(test_part[GROUP_COL]) == test_systems
        assert train_part["timestamp"].max() < test_part["timestamp"].min()
        assert set(train_part["timestamp"].unique()).isdisjoint(
            set(test_part["timestamp"].unique())
        )

        fitted = clone(estimator)
        fitted.fit(
            train_part.loc[:, feature_columns],
            train_part[TARGET_COL],
        )
        predictions = np.asarray(
            fitted.predict(test_part.loc[:, feature_columns]),
            dtype=float,
        )
        fold_metrics = normalized_regression_metrics(
            test_part[TARGET_COL], predictions
        )
        fold_rows.append({
            "fold": fold,
            "n_train_rows": len(train_part),
            "n_test_rows": len(test_part),
            "n_assigned_train_systems": len(assigned_train_systems),
            "n_train_systems_with_past_rows": len(actual_train_systems),
            "n_test_systems": len(test_systems),
            "train_end": train_part["timestamp"].max(),
            "test_start": test_part["timestamp"].min(),
            "observation_mean": float(test_part[TARGET_COL].mean()),
            "prediction_mean": float(predictions.mean()),
            **fold_metrics,
        })

        test_groups = test_part[GROUP_COL].reset_index(drop=True)
        test_observed = test_part[TARGET_COL].reset_index(drop=True).to_numpy()
        for station in sorted(test_systems):
            station_mask = test_groups.eq(station).to_numpy()
            if not station_mask.any():
                raise ValueError(
                    f"Held-out station {station!r} has no eligible future rows"
                )
            system_rows.append({
                "fold": fold,
                GROUP_COL: station,
                "n_test_rows": int(station_mask.sum()),
                **normalized_regression_metrics(
                    test_observed[station_mask],
                    predictions[station_mask],
                ),
            })
            tested_systems.append(station)

        del fitted, predictions, train_part, test_part

    assert set(tested_systems) == set(fold_by_station)
    assert len(tested_systems) == len(set(tested_systems))
    return {
        "fold_results": pd.DataFrame(fold_rows),
        "system_results": (
            pd.DataFrame(system_rows)
            .sort_values(GROUP_COL)
            .reset_index(drop=True)
        ),
    }


## Run models


In [ ]:
phase6_model_specs = {
    "training_mean": (
        DummyRegressor(strategy="mean"),
        [GHI_COL],
    ),
    "time_solar_only": (
        make_phase6_estimator(),
        PHASE6_TIME_ONLY_FEATURES,
    ),
    "model_a_weather": (
        make_phase6_estimator(),
        PHASE6_WEATHER_FEATURES,
    ),
    "model_c_weather_time": (
        make_phase6_estimator(),
        PHASE6_MODEL_C_FEATURES,
    ),
    "model_b_hybrid": (
        make_phase6_estimator(),
        PHASE6_HYBRID_FEATURES,
    ),
    "model_d_weather_time_physics": (
        make_phase6_estimator(),
        PHASE6_MODEL_D_FEATURES,
    ),
}

phase6_results = {}
for model_name in PHASE6_MODEL_ORDER:
    estimator, features = phase6_model_specs[model_name]
    print(f"Running {PHASE6_MODEL_LABELS[model_name]} ...")
    phase6_results[model_name] = evaluate_future_unseen_systems(
        df,
        estimator,
        features,
        phase6_fold_by_station,
    )

phase6_system_results = pd.concat(
    [
        phase6_results[model]["system_results"].assign(model=model)
        for model in PHASE6_MODEL_ORDER
    ],
    ignore_index=True,
).loc[
    :, ["model", "fold", GROUP_COL, "n_test_rows", "nRMSE", "MAE", "MBE"]
]
phase6_fold_results = pd.concat(
    [
        phase6_results[model]["fold_results"].assign(model=model)
        for model in PHASE6_MODEL_ORDER
    ],
    ignore_index=True,
)

phase6_pairing = phase6_system_results.pivot(
    index=GROUP_COL,
    columns="model",
    values=["fold", "n_test_rows"],
)
for field in ["fold", "n_test_rows"]:
    comparison = phase6_pairing[field].loc[:, PHASE6_MODEL_ORDER]
    assert comparison.eq(comparison.iloc[:, 0], axis=0).all().all()

phase6_summary = (
    phase6_system_results.groupby("model")[["nRMSE", "MAE", "MBE"]]
    .agg(["mean", "std", "median"])
)
phase6_summary.columns = [
    f"{metric}_{statistic}"
    for metric, statistic in phase6_summary.columns
]
phase6_summary = phase6_summary.reset_index()
phase6_summary.insert(
    1,
    "label",
    phase6_summary["model"].map(PHASE6_MODEL_LABELS),
)
phase6_summary["model"] = pd.Categorical(
    phase6_summary["model"],
    categories=PHASE6_MODEL_ORDER,
    ordered=True,
)
phase6_summary = phase6_summary.sort_values("model").reset_index(drop=True)
phase6_summary["model"] = phase6_summary["model"].astype(str)

PHASE6_RESULTS_DIR = EXPERIMENT_ROOT / "results"
PHASE6_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
phase6_split_audit.to_csv(
    PHASE6_RESULTS_DIR / "phase6_split_audit.csv", index=False
)
phase6_system_results.to_csv(
    PHASE6_RESULTS_DIR / "phase6_system_results.csv", index=False
)
phase6_fold_results.to_csv(
    PHASE6_RESULTS_DIR / "phase6_fold_results.csv", index=False
)
phase6_summary.to_csv(
    PHASE6_RESULTS_DIR / "phase6_summary.csv", index=False
)

print("Phase 6 Model D experiment complete. Results saved under results/.")
display(phase6_summary.round(4))


## Paired comparisons


In [ ]:
PHASE6_BOOTSTRAP_SAMPLES = 10_000

def phase6_paired_bootstrap(
    system_results,
    metric,
    reference_model,
    candidate_model,
    *,
    comparison,
    n_bootstrap=PHASE6_BOOTSTRAP_SAMPLES,
    random_state=PHASE6_RANDOM_STATE,
):
    paired = system_results.pivot(
        index=GROUP_COL,
        columns="model",
        values=metric,
    ).loc[:, [reference_model, candidate_model]]
    reference = paired[reference_model].to_numpy(dtype=float)
    candidate = paired[candidate_model].to_numpy(dtype=float)

    rng = np.random.default_rng(random_state)
    indices = rng.integers(
        0,
        len(paired),
        size=(n_bootstrap, len(paired)),
    )
    reference_bootstrap = reference[indices].mean(axis=1)
    candidate_bootstrap = candidate[indices].mean(axis=1)
    difference_bootstrap = candidate_bootstrap - reference_bootstrap

    result = {
        "comparison": comparison,
        "metric": metric,
        "reference_model": reference_model,
        "candidate_model": candidate_model,
        "n_systems": len(paired),
        "reference_macro_mean": reference.mean(),
        "candidate_macro_mean": candidate.mean(),
        "candidate_minus_reference": (candidate - reference).mean(),
        "difference_ci95_low": np.quantile(
            difference_bootstrap, 0.025
        ),
        "difference_ci95_high": np.quantile(
            difference_bootstrap, 0.975
        ),
        "candidate_better_systems": int(
            (candidate < reference).sum()
        ),
        "ties": int((candidate == reference).sum()),
        "relative_error_reduction_pct": np.nan,
        "relative_reduction_ci95_low": np.nan,
        "relative_reduction_ci95_high": np.nan,
    }
    if metric in {"nRMSE", "MAE"}:
        relative_bootstrap = (
            (reference_bootstrap - candidate_bootstrap)
            / reference_bootstrap
            * 100.0
        )
        result.update({
            "relative_error_reduction_pct": (
                (reference.mean() - candidate.mean())
                / reference.mean()
                * 100.0
            ),
            "relative_reduction_ci95_low": np.quantile(
                relative_bootstrap, 0.025
            ),
            "relative_reduction_ci95_high": np.quantile(
                relative_bootstrap, 0.975
            ),
        })
    return result

PHASE6_COMPARISON_SPECS = [
    ("hybrid_vs_weather", "model_a_weather", "model_b_hybrid"),
    ("time_control_vs_weather", "model_a_weather", "model_c_weather_time"),
    ("hybrid_vs_time_control", "model_c_weather_time", "model_b_hybrid"),
    (
        "model_d_vs_time_control",
        "model_c_weather_time",
        "model_d_weather_time_physics",
    ),
    (
        "model_d_vs_hybrid",
        "model_b_hybrid",
        "model_d_weather_time_physics",
    ),
]

phase6_paired_comparison = pd.DataFrame([
    phase6_paired_bootstrap(
        phase6_system_results,
        metric,
        reference_model,
        candidate_model,
        comparison=comparison,
    )
    for comparison, reference_model, candidate_model
    in PHASE6_COMPARISON_SPECS
    for metric in ["nRMSE", "MAE", "MBE"]
])
phase6_paired_comparison.to_csv(
    EXPERIMENT_ROOT / "results" / "phase6_paired_comparison.csv",
    index=False,
)
display(phase6_paired_comparison.round(4))


## Plot


In [ ]:
phase6_plot = phase6_summary.loc[
    :, ["model", "label", "nRMSE_mean"]
].copy()
colors = ["#7A8088", "#8E6BBE", "#3478B8", "#2A9D8F", "#D9693A", "#7B3F98"]

fig, axis = plt.subplots(figsize=(9.5, 5.4))
bars = axis.barh(
    phase6_plot["label"],
    phase6_plot["nRMSE_mean"],
    color=colors,
    edgecolor="#27313A",
    linewidth=0.8,
)
axis.invert_yaxis()
axis.set_xlabel("Macro nRMSE (fraction of installed capacity)")
axis.set_title(
    "Model D combined-feature experiment on unseen PV systems\n"
    "Train through 2022; test on held-out systems in 2023",
    loc="left",
)
axis.bar_label(bars, fmt="%.4f", padding=4)
axis.grid(axis="x", color="#D9DEE3", linewidth=0.8)
axis.set_axisbelow(True)
axis.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

phase6_figure_path = (
    EXPERIMENT_ROOT / "results" / "model_d_future_comparison.png"
)
fig.savefig(phase6_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved {phase6_figure_path}")
